# Inference Serving Lab: Why Splitting Prefill from Decode Doubled Throughput
Companion notebook for the Medium post *"The Two-Phase Machine"*.

Today we build three tiny things:
1. **A KV-cache calculator** — how many gigabytes a long prompt really costs in VRAM.
2. **A batching simulator** — static batching vs continuous batching (Orca-style), same traffic, honest utilization numbers.
3. **A speculative-decoding calculator** — expected speedup from a draft model, and why it fades at large batch sizes.

No GPU needed. Pure Python.

In [ ]:
import random
import math
import statistics
import matplotlib.pyplot as plt
random.seed(42)

## 1. The KV cache: decode is a memory problem

During decode, each token needs one attention lookup over the whole history. Instead of recomputing, engines cache the key/value vectors for every layer, head and past token. For a model with `L` layers, `kv_heads` KV heads of dimension `d_head`, stored in dtype of `b` bytes:

**bytes per token = 2 (K and V) × L × kv_heads × d_head × b**

Llama-3.1-8B: 32 layers, 8 KV heads (GQA), head dim 128, fp16 (2 bytes).

In [ ]:
def kv_bytes_per_token(layers, kv_heads, head_dim, bytes_per_elem=2):
    """Bytes of KV cache per token: 2 (K and V) x layers x kv_heads x head_dim x dtype bytes."""
    return 2 * layers * kv_heads * head_dim * bytes_per_elem

MODELS = {
    "Llama-3.1-8B":  dict(layers=32,  kv_heads=8,  head_dim=128),
    "Llama-3.1-70B": dict(layers=80,  kv_heads=8,  head_dim=128),
    "DeepSeek-V3 (MLA)": dict(layers=61, kv_heads=1, head_dim=512),  # latent: 1 x 512
}
for name, cfg in MODELS.items():
    per_tok = kv_bytes_per_token(**cfg)
    for ctx in (8_000, 32_000, 128_000):
        print(f"{name:18s} | ctx={ctx:>7,} -> {per_tok*ctx/1e9:6.2f} GB/request")
    print()

Look at the per-token numbers: on these configs Llama-3.1-8B's GQA (8 heads × 128 = 1024 dims/layer) and DeepSeek-V3's MLA (512-dim latent per layer) are within ~5% of each other per token. So what did MLA actually buy DeepSeek? The famous 32× figure is relative to **multi-head attention** — a 128-head MHA model would store 16,384 dims per layer; MLA's single 512-dim latent absorbs the full KV projection into one shared vector per token. For GQA models the practical win is different: the KV state is already small, and MLA's real edge is compressing the *query* side and sharing the latent across attention heads. Either way, the lesson stands: **decode is memory-bandwidth-bound, so per-token state is a first-class throughput lever** — which is exactly why PagedAttention, prefix caching, and KV-aware routing sit at the center of the serving stack.

## 2. Static batching vs continuous batching

A serving slot decodes one token per iteration. A *static* batch waits until every request in the batch finishes before admitting new ones (head-of-line blocking). *Continuous* batching (Orca, 2022) refills finished slots **every iteration**. Same traffic, same slots — measure slot utilization, mean completion time, and total makespan.

In [ ]:
def simulate(requests, n_slots=4, continuous=True):
    """requests: list of (arrival_iter, decode_len).
    Returns (makespan, mean_completion_time, slot_utilization)."""
    t = 0
    slots = [None] * n_slots          # each: [arrival, remaining]
    queue = sorted(requests)
    busy = 0
    completions = []

    def admit():
        for i in range(n_slots):
            if slots[i] is None:
                arrived = [r for r in queue if r[0] <= t]
                if arrived:
                    r = arrived[0]; queue.remove(r)
                    slots[i] = [r[0], r[1]]

    while queue or any(s is not None for s in slots):
        if continuous:
            admit()                       # refill finished slots every iteration
        elif not any(s is not None for s in slots):
            admit()                       # static: only admit when the whole batch drained
        # one decode iteration
        for i in range(n_slots):
            if slots[i] is not None:
                busy += 1
                slots[i][1] -= 1
                if slots[i][1] == 0:
                    completions.append(t + 1 - slots[i][0])
                    slots[i] = None
        t += 1

    makespan = t
    return makespan, statistics.mean(completions), busy / (n_slots * makespan)

In [ ]:
# Same workload for both: bursty arrivals, mixed lengths (one long request = head-of-line blocker)
random.seed(7)
workload = []
for i in range(12):
    arrival = random.randint(0, 6)
    length = 30 if i == 0 else random.randint(2, 8)
    workload.append((arrival, length))
print("workload (arrival, decode_len):", sorted(workload))

In [ ]:
for mode in (False, True):
    ms, mean_c, util = simulate(workload, n_slots=4, continuous=mode)
    print(f"{'CONTINUOUS' if mode else 'STATIC':10s} makespan={ms:3d}  mean_completion={mean_c:5.2f} iters  slot_util={util:.0%}")

**Read the numbers.** Continuous batching finishes the same 12 requests in far fewer iterations, with much higher slot utilization — no idle slots waiting on the long request's tail. In production this is one of the largest free wins: same GPU, strictly more throughput, at the cost of scheduler complexity. Try changing `n_slots` or the workload seed — the gap *widens* with more variable request lengths.

## 3. Speculative decoding: how much does a draft model buy you?

If the draft model accepts each proposed token independently with probability `p`, the expected number of tokens produced per target-model forward pass with draft depth `k` is the geometric sum `1 + p + p^2 + ... + p^k`. The draft pass isn't free, so real speedup is this divided by (1 + draft_cost_ratio).

In [ ]:
def expected_speedup(p, k, draft_cost=0.05):
    """p: per-token acceptance prob, k: draft depth, draft_cost: draft/target cost ratio."""
    accepted = sum(p ** i for i in range(k + 1))
    return accepted / (1 + draft_cost)

ps = [0.9, 0.8, 0.7, 0.6]
ks = list(range(1, 11))
plt.figure(figsize=(9, 4.5))
for p in ps:
    plt.plot(ks, [expected_speedup(p, k) for k in ks], marker="o", label=f"acceptance p={p}")
plt.axhline(1, color="gray", ls="--")
plt.xlabel("draft depth k"); plt.ylabel("expected speedup per forward pass")
plt.title("Speculative decoding: deeper drafts help — until acceptance decays")
plt.legend(); plt.grid(alpha=0.3); plt.show()

**Now the batch-size caveat from the post.** EAGLE-3's measured speedup on vLLM went 1.75× → 1.01× as batch size grew from 2 to 56. Model it: at high batch sizes the GPU is saturated, so each extra token of verification work costs real time — scale the benefit down by a saturation factor.

In [ ]:
batch_sizes = [2, 4, 8, 16, 24, 32, 48, 56]
measured    = [1.75, 1.68, 1.58, 1.49, 1.42, 1.36, 1.21, 1.01]  # vLLM + EAGLE-3, MT-Bench
plt.figure(figsize=(9, 4.5))
plt.plot(batch_sizes, measured, marker="o", lw=2, label="measured (vLLM + EAGLE-3)")
plt.axhline(1, color="gray", ls="--")
plt.xlabel("batch size"); plt.ylabel("speedup vs no speculation")
plt.title("Speculative decoding is a latency tool for low concurrency, not a saturation tool")
plt.legend(); plt.grid(alpha=0.3); plt.show()

## Takeaways

1. **Decode is a memory problem.** Per-token KV state (GBs at long context) is what bounds tokens/s — hence PagedAttention, MLA, and prefix caching.
2. **Scheduling is a free lunch.** Continuous batching beats static batching on the same hardware with the same traffic; the win scales with length variance.
3. **Speculative decoding is the only formally-identical speedup**, but it lives at low concurrency: acceptance compounds geometrically with depth, and the gains fade as the GPU saturates.

*Try it:* re-run section 2 with a 128k-context workload — watch slot utilization stay high while the KV calculator in section 1 tells you why VRAM, not compute, is now the binding constraint.